## CPSC Week 8 Assignment: Manipulating the Hypothesis Space

In this assgnment, I will train a Decision Tree on the "restaraunt wait" dataset. My objective is to see firsthand how a tree easily overfits its hypothesis space, and then i will use hyperparams to constrain that space and build a more logical model.

In [5]:
# import data as a panda df, lets see how some of the data looks like
# lets also see what kind of parameters we are working with

import pandas as pd
df = pd.read_csv('sample_data/restuarant_wait.csv')
print(df.head())

  Alternate  Bar FriSat Hungry Patrons Price Raining Reservation    Type  \
0       Yes   No     No    Yes    Some   $$$      No         Yes  French   
1       Yes   No     No    Yes    Full     $      No          No    Thai   
2        No  Yes     No     No    Some     $      No          No  Burger   
3       Yes   No    Yes    Yes    Full     $     Yes          No    Thai   
4       Yes   No    Yes     No    Full   $$$      No         Yes  French   

  WaitEstimate Wait  
0         0-10  Yes  
1        30-60   No  
2         0-10  Yes  
3        10-30  Yes  
4          >60   No  


In [6]:
# data preprocessing
import sklearn
from sklearn.model_selection import train_test_split


# Map Ordinal data (WaitEstimate)
wait_map = {'0-10': 0, '10-30': 1, '30-60': 2, '>60': 3}
df['WaitEstimate'] = df['WaitEstimate'].map(wait_map)

# defining features (X) and target (Y)
X = df.drop('Wait', axis=1)
y = df['Wait'].map({'Yes': 1, 'No': 0})

# 3. One-Hot Encode the rest (Alternate, Bar, FriSat, Hungry, Patrons, Price, Raining, Reservation, Type)
X = pd.get_dummies(X, drop_first=True)

# 4. Split for modeling
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Encoded Features: {X.columns.tolist()}")

Encoded Features: ['WaitEstimate', 'Alternate_Yes', 'Bar_Yes', 'FriSat_Yes', 'Hungry_Yes', 'Patrons_Some', 'Price_$$', 'Price_$$$', 'Raining_Yes', 'Reservation_Yes', 'Type_French', 'Type_Italian', 'Type_Thai']


In [9]:
from sklearn.tree import DecisionTreeClassifier, export_text

# here, we train the default tree w/ no hyperparameters
default_tree = DecisionTreeClassifier()
default_tree.fit(X_train, y_train)

# train the constrained tree, using training data
constrained_tree = DecisionTreeClassifier(max_depth=3, min_samples_split=10)
constrained_tree.fit(X_train, y_train)

# generate text output
feature_list = list(X.columns)

# DEFAULT TREE, trained on x_train.
# This tree has no rules or hyperparameters
print("DEFAULT TREE")
print(export_text(default_tree, feature_names=feature_list))

# CONSTRAINED TREE, trained on x train

"""
Adding max_depth=3 and min_samples_split=10 gives pruning,
preventing overfitting, removing random parameters like type_italian
from our descision tree
"""

print("\n\n")
print("CONSTRAINED TREE")
print(export_text(constrained_tree, feature_names=feature_list))

DEFAULT TREE
|--- Patrons_Some <= 0.50
|   |--- WaitEstimate <= 1.50
|   |   |--- WaitEstimate <= 0.50
|   |   |   |--- class: 0
|   |   |--- WaitEstimate >  0.50
|   |   |   |--- Price_$$$ <= 0.50
|   |   |   |   |--- class: 1
|   |   |   |--- Price_$$$ >  0.50
|   |   |   |   |--- Type_Italian <= 0.50
|   |   |   |   |   |--- class: 1
|   |   |   |   |--- Type_Italian >  0.50
|   |   |   |   |   |--- class: 0
|   |--- WaitEstimate >  1.50
|   |   |--- Bar_Yes <= 0.50
|   |   |   |--- class: 0
|   |   |--- Bar_Yes >  0.50
|   |   |   |--- WaitEstimate <= 2.50
|   |   |   |   |--- Raining_Yes <= 0.50
|   |   |   |   |   |--- Reservation_Yes <= 0.50
|   |   |   |   |   |   |--- class: 1
|   |   |   |   |   |--- Reservation_Yes >  0.50
|   |   |   |   |   |   |--- class: 0
|   |   |   |   |--- Raining_Yes >  0.50
|   |   |   |   |   |--- class: 0
|   |   |   |--- WaitEstimate >  2.50
|   |   |   |   |--- class: 0
|--- Patrons_Some >  0.50
|   |--- class: 1




CONSTRAINED TREE
|--- Patro